# Lab 4: The Average Price Elasticity

> Requires the artifact written by **Lab 2**.

Lab 1 regressed quantity on price and got $\hat\delta \approx -0.2$, which amounts to
almost no price response. Lab 3 showed why that estimate is suspect and what to condition
on instead. This notebook estimates the elasticity properly.

## The causal model

**Assumption 1 (structural equation model).** In each period,

$$Q_{it} = a_t(S_{it}, \epsilon_{it})\,P_{it} + q_t(S_{it}, \epsilon_{it}), \qquad
P_{it} = p_t(S_{it}, \epsilon^p_{it}), \qquad
S_{it} = s_t(S_{i,t-1}, \epsilon^s_{it}),$$

with $a_t, q_t, p_t, s_t$ nonparametric and the shocks mutually independent. The state is

$$S_{it} = (Q_{i,t-1},\; P_{i,t-1},\; X_{it}),$$

lagged quantity and price together with product characteristics $X_{it}$, meaning the
embeddings from Lab 2 plus the tabular controls.

**The role of the lagged outcome.** $Q_{i,t-1}$ summarises everything that made the
product sell last month: visibility in search results, accumulated reviews, brand
strength, perceived quality. These are the same latent factors that let a seller sustain a
higher price. Lab 3 showed the point concretely, since once $Q_{i,t-1}$ is in the model
the embeddings add almost nothing to the prediction of $Q_{it}$.

**The causal parameter.** Writing $A_{it} := a_t(S_{it}, \epsilon_{it})$, the potential
outcome at price $p$ is $Q_{it}(p) = A_{it}\,p + q_t(S_{it}, \epsilon_{it})$, so $A_{it}$
is the causal effect of a one-unit increase in log price. This notebook estimates its
average, $\alpha_t = E[A_{it}]$. Lab 5 estimates how it varies.

## Estimation by partialling out

Conditioning on $S_{it}$ blocks the non-causal paths and gives the projection equation

$$Q^{\perp}_{it} = \delta\,P^{\perp}_{it} + e_{it}, \qquad
Q^{\perp}_{it} = Q_{it} - E[Q_{it}\mid S_{it}], \quad
P^{\perp}_{it} = P_{it} - E[P_{it}\mid S_{it}],$$

which we estimate by double machine learning: learn the two conditional expectations with
cross-fitted machine learning, then regress residual on residual.

Neyman orthogonality is what allows this to work with estimated nuisance functions, since
small errors in $E[Q\mid S]$ and $E[P\mid S]$ have no first-order effect on $\hat\delta$.
It is also why estimated embeddings do not invalidate the inference, provided they were
fitted on independent data. They were: Lab 2 fine-tuned on the fine-tune products, and
everything below runs on the estimation products only.

Standard errors are clustered at the product level throughout, since each product
contributes 13 correlated observations.

## Setup

In [ ]:
%pip install -q doubleml lightgbm

In [ ]:
import os
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import statsmodels.api as sm
from doubleml import DoubleMLData, DoubleMLPLR
from lightgbm import LGBMRegressor
from sklearn.metrics import r2_score

palette = sns.color_palette("colorblind")
warnings.filterwarnings("ignore", message="X does not have valid feature names")
pd.set_option("display.width", 140)

CONFIDENCE = 0.90
RANK_TO_DEMAND = 2.0   # 1 / theta-hat, theta-hat ~ 0.5 for toys (He & Hollenbeck)

## Loading the artifact from Lab 2

In [ ]:
ARTIFACT = "subsample_v2.parquet"
drive_path = f"/content/drive/MyDrive/demand_labs_v2/{ARTIFACT}"

# Colab gives every notebook its own machine, so /content does not carry across
# labs. Google Drive does.
if os.path.isdir("/content") and not os.path.exists(drive_path):
    try:
        from google.colab import drive
        drive.mount("/content/drive")
    except Exception as exc:
        print(f"could not mount Drive ({type(exc).__name__})")

path = next((p for p in [drive_path, f"/content/{ARTIFACT}", ARTIFACT]
             if os.path.exists(p)), None)
if path is None:
    raise FileNotFoundError(
        "subsample_v2.parquet not found. Run Lab 2 "
        "(02_finetune_embeddings.ipynb) first and let it save to Google Drive, "
        "or upload the file into this session.")

df = pd.read_parquet(path)
print(f"loaded {path}")
print(f"{df['ASIN'].nunique():,} products, {len(df):,} rows, "
      f"{df['period'].nunique()} periods")
print(df.groupby("split").agg(products=("ASIN", "nunique"), rows=("ASIN", "size")))

In [ ]:
emb_cols = [c for c in df.columns if c.startswith("emb_")]
pca_cols = [c for c in df.columns if c.startswith("pca_")]
sim_cols = [c for c in df.columns if c.startswith("similarity_cluster_")]
sub_cols = [c for c in df.columns if c.startswith("sub_")]

lag_controls = ["Q_t-1", "P_bb_t-1"]
tab_controls = ["RATING_t-1", "REVIEW_COUNT_t-1", "n_offers",
                "n_offers_fba", "n_offers_fbm", "lightning_deal", "is_fba"]

periods = sorted(df["period"].unique())[1:]   # first period is the baseline
period_cols = [f"period_{p}" for p in periods]
for p in periods:
    df[f"period_{p}"] = (df["period"] == p).astype(int)

OUTCOME, TREATMENT = "Q_t", "P_bb_t"

# The paper's I_2: estimate only on products the embeddings were not trained on.
data = (df[df["split"] == "estimation"]
        .dropna(subset=lag_controls + tab_controls + [OUTCOME, TREATMENT])
        .reset_index(drop=True))

print(f"estimation sample: {len(data):,} observations, "
      f"{data['ASIN'].nunique()} products, {data['period'].nunique()} periods")

## Control specifications

These map onto Table 7 of the paper. $X^o_{it}$ denotes the tabular controls (ratings,
review counts, offer counts, deal and FBA flags) together with the subcategory and period
dummies.

| Label | Control function $\gamma_t(S_{it})$ |
|---|---|
| **I-1** Linear $(P_{t-1},Q_{t-1})$ | lagged price and quantity only |
| **I-1** Linear $(P_{t-1},Q_{t-1},X^e,X^o)$ | plus the 256 embeddings |
| **I-1** Linear $(P_{t-1},Q_{t-1},X^{sim},X^o)$ | plus the 5 cluster similarities |
| **I-2** Linear with interactions | plus interactions of $P_{t-1},Q_{t-1}$ with $X^{sim}$ |
| **I-3** Boosted trees $(\cdot,X^e,X^o)$ | fully nonparametric, embeddings |
| **I-3** Boosted trees $(\cdot,X^{sim},X^o)$ | fully nonparametric, similarities |

If the estimate barely moves across this list, the controls are doing their job and the
remaining variation in price is plausibly unrelated to demand.

In [ ]:
X_o = tab_controls + sub_cols + period_cols

# I-2: interactions of the lagged state with the cluster similarities
inter_cols = []
for lag in lag_controls:
    for s in sim_cols:
        name = f"int_{lag}_{s}"
        data[name] = data[lag] * data[s]
        inter_cols.append(name)

SPECS = {
    "I-1 Linear (P,Q lags)":        (lag_controls, "linear"),
    "I-1 Linear (+ embeddings)":    (lag_controls + emb_cols + X_o, "linear"),
    "I-1 Linear (+ similarities)":  (lag_controls + sim_cols + X_o, "linear"),
    "I-2 Linear (+ interactions)":  (lag_controls + sim_cols + X_o + inter_cols, "linear"),
    "I-3 Boosted (+ embeddings)":   (lag_controls + emb_cols + X_o, "dml"),
    "I-3 Boosted (+ similarities)": (lag_controls + sim_cols + X_o, "dml"),
}
print(f"{len(inter_cols)} interaction terms; {len(X_o)} tabular/dummy controls")

## Linear specifications

Ordinary least squares with the control function entered linearly, and cluster-robust
standard errors.

In [ ]:
def lin_model(frame, treatment, controls, outcome=OUTCOME, level=CONFIDENCE):
    """OLS of outcome on treatment + controls, clustered by product."""
    X = sm.add_constant(frame[[treatment] + controls], has_constant="add")
    res = sm.OLS(frame[outcome], X).fit(
        cov_type="cluster", cov_kwds={"groups": frame["ASIN"]})
    ci = res.conf_int(alpha=1 - level)
    return {
        "coef": res.params[treatment],
        "se": res.bse[treatment],
        "ci_lo": ci.loc[treatment, 0],
        "ci_hi": ci.loc[treatment, 1],
        "pval": res.pvalues[treatment],
    }


results = {}
for label, (controls, kind) in SPECS.items():
    if kind != "linear":
        continue
    results[label] = lin_model(data, TREATMENT, controls)
    r = results[label]
    print(f"{label:32s} delta={r['coef']:+.3f}  "
          f"[{r['ci_lo']:+.3f}, {r['ci_hi']:+.3f}]  se={r['se']:.3f}")

## Double machine learning

For the nonparametric control function we hand both nuisance regressions to boosted trees
and let `DoubleML` handle the cross-fitting. Passing `cluster_cols="ASIN"` keeps all
observations of a product inside the same fold, which prevents a product's own past from
leaking across the split, and produces cluster-robust standard errors.

In [ ]:
def _r2(y_true, y_pred):
    """R^2 metric in the signature DoubleML.evaluate_learners expects."""
    y_true, y_pred = np.asarray(y_true).ravel(), np.asarray(y_pred).ravel()
    ok = np.isfinite(y_true) & np.isfinite(y_pred)
    return r2_score(y_true[ok], y_pred[ok])


def dml_plr(frame, controls, label, n_folds=3, level=CONFIDENCE):
    """Partially linear DML with cluster-aware cross-fitting."""
    cols = [OUTCOME, TREATMENT, "ASIN"] + controls
    dml_data = DoubleMLData(frame[cols], y_col=OUTCOME, d_cols=TREATMENT,
                            x_cols=controls, cluster_cols="ASIN")

    learner = dict(n_estimators=500, learning_rate=0.02, random_state=42, verbose=-1)
    obj = DoubleMLPLR(dml_data,
                      ml_l=LGBMRegressor(**learner),
                      ml_m=LGBMRegressor(**learner),
                      score="partialling out", n_folds=n_folds)
    obj.fit(store_predictions=True)

    ci = obj.confint(level=level)
    scores = obj.evaluate_learners(metric=_r2)
    return obj, {
        "coef": float(obj.coef[0]), "se": float(obj.se[0]),
        "ci_lo": float(ci.iloc[0, 0]), "ci_hi": float(ci.iloc[0, 1]),
        "pval": float(obj.pval[0]),
        "r2_l": float(np.mean(scores["ml_l"])),
        "r2_m": float(np.mean(scores["ml_m"])),
    }


dml_objects = {}
for label, (controls, kind) in SPECS.items():
    if kind != "dml":
        continue
    np.random.seed(3141)
    obj, res = dml_plr(data, controls, label)
    dml_objects[label], results[label] = obj, res
    print(f"{label:32s} delta={res['coef']:+.3f}  "
          f"[{res['ci_lo']:+.3f}, {res['ci_hi']:+.3f}]  "
          f"R2(Q|S)={res['r2_l']:.3f}  R2(P|S)={res['r2_m']:.3f}")

## Results

In [ ]:
table = pd.DataFrame(results).T[["coef", "se", "ci_lo", "ci_hi", "pval"]]
table["demand elasticity"] = table["coef"] * RANK_TO_DEMAND
table.index.name = "Specification"
table.round(3)

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4.5))
labels = list(table.index)
y = np.arange(len(labels))
colors = [palette[0] if lbl.startswith("I-3") else palette[2] for lbl in labels]

ax.errorbar(table["coef"], y,
            xerr=[table["coef"] - table["ci_lo"], table["ci_hi"] - table["coef"]],
            fmt="o", ms=7, lw=1.6, capsize=4, capthick=1.6,
            ecolor="gray", linestyle="none", color="none")
ax.scatter(table["coef"], y, c=colors, s=55, zorder=3)
ax.axvline(0, color="gray", ls="--", lw=1)
ax.set_yticks(y)
ax.set_yticklabels(labels, fontsize=9)
ax.invert_yaxis()
ax.set_xlabel("$\\hat{\\delta}$: effect of log price on the log inverse sales rank")
ax.set_title(f"Average price effect, {CONFIDENCE:.0%} confidence intervals")
ax.grid(axis="x", alpha=0.35)
plt.tight_layout()
plt.show()

## Diagnostics

The estimator is a regression of one residual on another, so the residuals are the
quickest way to check that the identifying variation is present.

In [ ]:
label = "I-3 Boosted (+ similarities)"
obj = dml_objects[label]
Q_perp = data[OUTCOME].values - obj.predictions["ml_l"][:, 0, 0]
P_perp = data[TREATMENT].values - obj.predictions["ml_m"][:, 0, 0]
delta_hat = results[label]["coef"]

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

axes[0].hist(Q_perp, bins=60, alpha=0.75, color=palette[0],
             label="$\\hat{Q}^{\\perp}_{it}$")
axes[0].hist(P_perp, bins=60, alpha=0.75, color=palette[1],
             label="$\\hat{P}^{\\perp}_{it}$")
axes[0].set_xlabel("residual")
axes[0].set_title("Partialled-out residuals")
axes[0].legend()

axes[1].scatter(P_perp, Q_perp, s=7, alpha=0.15, color=palette[0])
grid = np.linspace(P_perp.min(), P_perp.max(), 100)
axes[1].plot(grid, delta_hat * grid, color=palette[1], lw=2,
             label=f"$\\hat{{\\delta}} = {delta_hat:.3f}$")
axes[1].set_xlabel("$\\hat{P}^{\\perp}_{it}$")
axes[1].set_ylabel("$\\hat{Q}^{\\perp}_{it}$")
axes[1].set_title(f"Second stage: {label}")
axes[1].legend()

for ax in axes:
    ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

print(f"sd of price residual:    {P_perp.std():.4f}")
print(f"sd of quantity residual: {Q_perp.std():.4f}")

## What we found

**The elasticity is negative, sizeable and stable across specifications.** In the paper's
full sample the confidence intervals sit between $-0.79$ and $-0.54$. Multiplying by the
Pareto factor of about 2 puts the demand elasticity near $[-1.6, -1.1]$, so a 1% price
rise costs roughly 1 to 1.6% of units sold. That is an ordinary figure for a consumer good
with close substitutes, and far from the $-0.2$ of Lab 1.

**Almost all of the correction comes from the lagged state.** Compare the first row with
the rest: adding embeddings, similarities, interactions or boosted trees on top of
$(Q_{t-1}, P_{t-1})$ moves the estimate very little. Lab 3 anticipated this. The
embeddings explain levels well and changes poorly, and it is the changes that identify
$\delta$, so they cannot be absorbing much confounding.

**The stability is itself informative.** An estimate that moved as controls were added
would suggest the control function was still picking up confounding. One that does not
move suggests the lagged state has already accounted for it.

### What this does not establish

Identification requires that no time-varying unobserved shock moves both price and
quantity within a period. A demand shock that a seller responds to by repricing would
violate it. The paper's defence is empirical: prices are sticky, and Lab 1 found a third
of product-periods with an exactly unchanged price, so the link from this period's demand
shock to this period's price is likely weak. The paper also reports a sensitivity
analysis, in which a confounder would have to explain about 15% of the residual variation
in both price and quantity to push the estimate to zero, against a benchmark where the
remaining observed covariates explain essentially none.

### Next

The average conceals a good deal of variation. Table 8 of the paper shows the elasticity
ranging from about $-2.0$ to $+0.4$ across products, depending on where a product sits in
the embedding space and on how popular and expensive it is. Lab 5 estimates that
variation.

> **A note on your numbers.** The estimation sample here is around 6,500 observations
> against the paper's 38,041, so expect confidence intervals roughly twice as wide as the
> published ones. The point estimate should still land clearly below zero, and the
> boosted-tree rows should come out near $-0.7$.